# 13 â€” Summary & Consolidated Pipeline
Writes `reports/results_summary.md`; consolidated runnable pipeline lives in `src/pipeline.py`.

In [1]:

import sys, os
from pathlib import Path
root = Path.cwd()
while not (root / "src").exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))
%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")

In [2]:
import json
from pathlib import Path
import pandas as pd
from src.config import RESULTS_DIR

val = json.load(open(RESULTS_DIR / "validation_report.json"))
final = json.load(open(RESULTS_DIR / "final_test_results.json"))
lb_r = pd.read_csv(RESULTS_DIR / "leaderboard_regression.csv")
lb_c = pd.read_csv(RESULTS_DIR / "leaderboard_classification.csv")

lines = ["# Results Summary", "",
         f"- Rows kept after validation: {val['rows_kept']} (dropped {val['rows_dropped']})",
         "- i.i.d. assumption: no user id/timestamp column exists; if rows are repeated measures, mild leakage could inflate CV slightly.",
         "", "## Regression leaderboard (CV MAE, lower is better)", "",
         lb_r.sort_values(["target", "mae_mean"]).to_markdown(index=False), "",
         "## Classification leaderboard (CV PR-AUC, higher is better)", "",
         lb_c.sort_values("pr_auc_mean", ascending=False).to_markdown(index=False), "",
         "## Final held-out test results", ""]
nested = {}
for tk, v in final.items():
    t, k = tk.split('.', 1)
    nested.setdefault(t, {})[k] = v
final = nested
for t, m in final.items():
    pretty = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in m.items()}
    lines += [f"### {t}", "", "```json", json.dumps(pretty, indent=2), "```", ""]
lines += ["## Notes", "",
          "- Test set evaluated exactly once (notebook 12).",
          "- SVM tuning used a fixed 5000-row subsample for tractability.",
          "- Class imbalance handled via class weights / scale_pos_weight / is_unbalance; PR-AUC is the primary classification metric."]
summary = "\n".join(lines)
Path("..", "reports", "results_summary.md").write_text(summary, encoding="utf-8")
print(summary[:1500])


# Results Summary

- Rows kept after validation: 20000 (dropped 0)
- i.i.d. assumption: no user id/timestamp column exists; if rows are repeated measures, mild leakage could inflate CV slightly.

## Regression leaderboard (CV MAE, lower is better)

| model                | target             |   mae_mean |   rmse_mean |   r2_mean |
|:---------------------|:-------------------|-----------:|------------:|----------:|
| random_forest_tuned  | next_cycle_length  |    1.1916  |     1.73416 |   0.79573 |
| xgboost_tuned        | next_cycle_length  |    1.20509 |     1.75389 |   0.79109 |
| lightgbm_tuned       | next_cycle_length  |    1.21017 |     1.76233 |   0.78908 |
| random_forest        | next_cycle_length  |    1.21759 |     1.77032 |   0.78722 |
| ridge_tuned          | next_cycle_length  |    1.22316 |     1.78756 |   0.78301 |
| ridge                | next_cycle_length  |    1.22345 |     1.78799 |   0.7829  |
| svr_tuned            | next_cycle_length  |    1.23922 |     1.78624 

## Verify consolidated pipeline (`src/pipeline.py`)

In [3]:
from pathlib import Path
p = Path("..") / "src" / "pipeline.py"
print(p.resolve(), "exists:", p.exists())
print(p.read_text(encoding="utf-8")[:600] if p.exists() else "pipeline.py will be created next")


E:\NAVYA_MODEL\src\pipeline.py exists: True
import json

import joblib
import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import (
    FEATURES, MODELS_DIR, RESULTS_DIR, SEED, SPLITS_DIR,
    TARGET_CLF, TARGETS_REG, TEST_SIZE, VAL_SIZE,
)
from src.evaluate import clf_metrics, confusion, reg_metrics
from src.features import engineer
from src.preprocessing import build_preprocessor
from src.split import make_splits
from src.tune import make_est
from src.validation import clean, load_raw, validate


def load_and_prepare():
    raw = load_raw()
    report = validate(raw)
    cleaned, _ = clean(raw, re
